### TRANSFORMATION WORKFLOW

1. TRIM SPACES
2. RENAME COLUMNS
3. EXTRACT CATEGORY_ID AND PRODUCT_KEY_SHORT
4. NORMALIZE PRODUCT_LINE
5. HANDLE NULL PRODUCT_COST
6. WRITE INTO SILVER

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
# 0) LOAD DATA & READ FROM BRONZE
dfprod = spark.table("acdproj.bronze.crm_prd_info")

In [0]:
# 1) TRIM SPACES
for field in dfprod.schema.fields:
    if isinstance(field.dataType, StringType):
        dfprod = dfprod.withColumn(field.name, F.trim(F.col(field.name)))

In [0]:
# 2) RENAME COLUMNS
RENAME_MAP_PROD = {
    "prd_id": "product_id",
    "prd_key": "product_key",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "product_start_date",
    "prd_end_dt": "product_end_date"
}
for old_name, new_name in RENAME_MAP_PROD.items():
    dfprod = dfprod.withColumnRenamed(old_name, new_name)

In [0]:
# 3) EXTRACT CATEGORY_ID AND PRODUCT_KEY_SHORT
dfprod = dfprod.withColumn("category_id", F.substring(F.col("product_key"), 1, 5))
dfprod = dfprod.withColumn("product_key_short", F.substring(F.col("product_key"), 7, 100))

In [0]:
# 4) NORMALIZE PRODUCT_LINE
dfprod = dfprod.withColumn(
    "product_line",
    F.when(F.upper(F.col("product_line")) == "S", "Sales")
     .when(F.upper(F.col("product_line")) == "M", "Mountain")
     .when(F.upper(F.col("product_line")) == "R", "Road")
     .when(F.upper(F.col("product_line")) == "T", "Touring")
     .otherwise("n/a")
)


In [0]:
# 5) HANDLE NULL PRODUCT_COST
dfprod = dfprod.withColumn("product_cost", F.coalesce(F.col("product_cost"), F.lit(0)))

In [0]:
# 6) WRITE INTO SILVER
dfprod.write.mode("overwrite").saveAsTable("acdproj.silver.crm_products")